In [0]:
%sql
CREATE OR REPLACE TABLE banking.gold.branch_performance AS
WITH customer_branch AS (
    SELECT customer_id, branch_code
    FROM banking.silver.customers
),
account_agg AS (
    SELECT
        customer_id,
        COUNT(account_id) AS total_accounts,
        SUM(balance)      AS total_balance
    FROM banking.silver.accounts
    GROUP BY customer_id
),
txn_agg AS (
    SELECT
        a.customer_id,
        COUNT(t.txn_id) AS total_transactions,
        SUM(t.amount)   AS total_transaction_amount
    FROM banking.silver.accounts a
    JOIN banking.silver.transactions t
        ON t.account_id = a.account_id
    GROUP BY a.customer_id
)
SELECT
    b.branch_code,
    b.branch_name,
    COUNT(DISTINCT cb.customer_id)             AS total_customers,
    COALESCE(SUM(a.total_accounts), 0)         AS total_accounts,
    COALESCE(SUM(a.total_balance), 0)          AS total_deposits,
    COALESCE(SUM(t.total_transactions), 0)     AS total_transactions,
    COALESCE(SUM(t.total_transaction_amount), 0) AS total_transaction_amount
FROM banking.silver.branches b
LEFT JOIN customer_branch cb ON b.branch_code = cb.branch_code
LEFT JOIN account_agg a      ON cb.customer_id = a.customer_id
LEFT JOIN txn_agg t          ON cb.customer_id = t.customer_id
GROUP BY b.branch_code, b.branch_name

In [0]:

count=spark.sql("""
                SELECT COUNT(*)AS cnt
                FROM banking.gold.branch_performance
                """).collect()[0]["cnt"]

dbutils.notebook.exit(str(count))
print(count)

In [0]:
%sql
create schema if not exists banking.gold